# Article-presentation figures: risk-averse single-allocation hub location

This executed notebook produces separate 300-DPI academic figures using the implemented 100-scenario setup. Exact MILP is used only for tiny validation; all CAB25/AP charts are labelled **heuristic/GVNS**, not optimal.

## Experiment design

**Scenario count:** 100 equally weighted scenarios per run. **CAB25 parameter sweeps:** scenario seed 0, search seed 0, `max_iterations=3`, `max_evaluations=8`. **Repeated heuristic comparison:** CAB25, AP100, AP150 and AP200; paired scenario/search seeds 0 and 1; `p=3`, `alpha=0.5`, `beta=0.5`. GVNS/DL-GVNS and CBS/DL-CBS use the same immutable scenario bundle within each paired run. Benders uses the same 100 scenarios but reports an iteration/time-bounded incumbent and certified master bounds; it is not labelled optimal unless the gap closes.

In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

from single_allocation_hub_location.data import load_matrix_pair
from single_allocation_hub_location.experiments import DATASETS
from single_allocation_hub_location.gvns import GVNSConfig, solve_gvns, solve_dl_gvns, load_rank_scores
from single_allocation_hub_location.benders import solve_benders
from single_allocation_hub_location.cbs import solve_cbs, solve_dl_cbs
from single_allocation_hub_location.gvns_experiments import (
    run_small_grid,
    train_synthetic_ranker_checkpoint,
)
from single_allocation_hub_location.scenarios import generate_flow_scenarios

plt.style.use("seaborn-v0_8-whitegrid")

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data/raw").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from the repository or a child directory.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data/raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs/article_presentation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIO_COUNT = 100
CAB25_SCENARIO_SEED = 0
CAB25_SEARCH_SEED = 0
SWEEP_ITERATIONS = 3
SWEEP_EVALUATIONS = 8
COMPARISON_SEEDS = (0, 1)
COMPARISON_ITERATIONS = 3
COMPARISON_EVALUATIONS = 8
RANKER_CHECKPOINT = OUTPUT_DIR / "synthetic_ranker.pt"

assert SCENARIO_COUNT == 100
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /home/miladkahnooji/Milad/Personal_projects/karlancer/projects/Python math model/Single-allocation-hub-location/outputs/article_presentation


## Synthetic-only ranker checkpoint

DL-GVNS needs the existing compact DLHr ranker. The checkpoint below is trained only on the project’s synthetic data; no CAB/AP evaluation labels are used for training.

In [2]:
ranker_identity = train_synthetic_ranker_checkpoint(
    RANKER_CHECKPOINT, training_instance_count=6, seed=0
)
print(ranker_identity)
print(RANKER_CHECKPOINT)

synthetic_training:6:seed=0
/home/miladkahnooji/Milad/Personal_projects/karlancer/projects/Python math model/Single-allocation-hub-location/outputs/article_presentation/synthetic_ranker.pt


## CAB25 objective sensitivity to the risk parameter

The curves use baseline seeded GVNS on identical 100 scenarios. They are heuristic results, not proofs of optimality.

In [3]:
flow_name, distance_name = DATASETS["CAB25"]
cab25 = load_matrix_pair(DATA_ROOT / flow_name, DATA_ROOT / distance_name)
cab25_distance = np.asarray(cab25.distance)
cab25_demand = np.asarray(cab25.flow)
cab25_scenarios = generate_flow_scenarios(
    cab25_demand, scenario_count=SCENARIO_COUNT, seed=CAB25_SCENARIO_SEED
)

def cab25_gvns(p: int, alpha: float, beta: float) -> float:
    result = solve_gvns(
        cab25_distance,
        cab25_scenarios.flows,
        cab25_scenarios.probabilities,
        GVNSConfig(
            p=p, alpha=alpha, beta=beta,
            max_iterations=SWEEP_ITERATIONS,
            max_evaluations=SWEEP_EVALUATIONS,
        ),
        CAB25_SEARCH_SEED,
    )
    assert result.proven_optimal is False
    return result.objective

betas = (0.1, 0.3, 0.5, 0.8)
beta_values = {p: [cab25_gvns(p, 0.5, beta) for beta in betas] for p in (2, 3, 4, 5)}

fig, ax = plt.subplots(figsize=(7.2, 4.6))
for p, values in beta_values.items():
    ax.plot(betas, values, marker="o", linewidth=2, label=f"p = {p}")
ax.set_title("CAB25: Heuristic Objective versus Risk Parameter")
ax.set_xlabel(r"Risk parameter $\beta$ (worst-tail fraction)")
ax.set_ylabel("Conditional beta-mean objective")
ax.legend(title="Hub count")
ax.text(0.02, 0.02, "GVNS heuristic; 100 scenarios; budget = 8 evaluations", transform=ax.transAxes, fontsize=8)
fig.tight_layout()
beta_path = OUTPUT_DIR / "cab25_objective_vs_beta.png"
fig.savefig(beta_path, dpi=300, bbox_inches="tight")
plt.show()
print(beta_path)

/home/miladkahnooji/Milad/Personal_projects/karlancer/projects/Python math model/Single-allocation-hub-location/outputs/article_presentation/cab25_objective_vs_beta.png


/tmp/ipykernel_1170763/2390142390.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## CAB25 objective sensitivity to the inter-hub discount

The same 100 scenarios and seeded GVNS budget are held fixed while varying alpha.

In [4]:
alphas = (0.2, 0.5, 0.8)
alpha_values = {p: [cab25_gvns(p, alpha, 0.5) for alpha in alphas] for p in (2, 3, 4, 5)}

fig, ax = plt.subplots(figsize=(7.2, 4.6))
for p, values in alpha_values.items():
    ax.plot(alphas, values, marker="o", linewidth=2, label=f"p = {p}")
ax.set_title("CAB25: Heuristic Objective versus Inter-hub Discount")
ax.set_xlabel(r"Inter-hub discount $\alpha$")
ax.set_ylabel("Conditional beta-mean objective")
ax.legend(title="Hub count")
ax.text(0.02, 0.02, "GVNS heuristic; 100 scenarios; budget = 8 evaluations", transform=ax.transAxes, fontsize=8)
fig.tight_layout()
alpha_path = OUTPUT_DIR / "cab25_objective_vs_alpha.png"
fig.savefig(alpha_path, dpi=300, bbox_inches="tight")
plt.show()
print(alpha_path)

/home/miladkahnooji/Milad/Personal_projects/karlancer/projects/Python math model/Single-allocation-hub-location/outputs/article_presentation/cab25_objective_vs_alpha.png


/tmp/ipykernel_1170763/2693023182.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Paired GVNS and DL-GVNS comparison

For every dataset/seed pair, baseline GVNS and DL-GVNS receive identical generated flows, probabilities, scenario seed, search seed, p, alpha, beta, and evaluation budget. The ranker checkpoint is synthetic-only.

In [5]:
records = run_small_grid(
    datasets=("CAB25", "AP100", "AP150", "AP200"),
    seeds=COMPARISON_SEEDS,
    p=3,
    alpha=0.5,
    beta=0.5,
    max_iterations=COMPARISON_ITERATIONS,
    max_evaluations=COMPARISON_EVALUATIONS,
    ranker_checkpoint=RANKER_CHECKPOINT,
    data_root=DATA_ROOT,
)

assert len(records) == 16
assert all(record.scenario_count == 100 and record.proven_optimal is False for record in records)
assert all(np.isfinite(record.objective) and np.isfinite(record.runtime) for record in records)

datasets = ("CAB25", "AP100", "AP150", "AP200")
methods = ("gvns", "dl_gvns")
labels = {"gvns": "GVNS", "dl_gvns": "DL-GVNS"}

def mean_and_std(field: str, dataset: str, method: str) -> tuple[float, float]:
    values = np.array([getattr(record, field) for record in records if record.dataset == dataset and record.method == method])
    return float(values.mean()), float(values.std(ddof=1))

objective_summary = {
    (dataset, method): mean_and_std("objective", dataset, method)
    for dataset in datasets for method in methods
}
runtime_summary = {
    (dataset, method): mean_and_std("runtime", dataset, method)
    for dataset in datasets for method in methods
}
for dataset in datasets:
    print(dataset, {
        labels[method]: {
            "objective": objective_summary[(dataset, method)],
            "runtime": runtime_summary[(dataset, method)],
        }
        for method in methods
    })

CAB25 {'GVNS': {'objective': (1237.0304133793375, 78.1373926233314), 'runtime': (0.001915227499921457, 9.930678312446061e-05)}, 'DL-GVNS': {'objective': (1233.9910874472887, 7.242273587912664), 'runtime': (0.0015685309999753372, 7.18561836347149e-06)}}
AP100 {'GVNS': {'objective': (23.877464954950483, 1.6019639435625472), 'runtime': (0.048074386500047694, 0.009818070882741918)}, 'DL-GVNS': {'objective': (21.819380697890907, 0.22498673425845306), 'runtime': (0.01883362650005438, 0.00020606859362714473)}}
AP150 {'GVNS': {'objective': (24.581235342046753, 0.851525401331408), 'runtime': (0.03753551650015652, 0.0012370896780335577)}, 'DL-GVNS': {'objective': (21.159172748361698, 0.13780410875198854), 'runtime': (0.03399774450008408, 0.001555564914102667)}}
AP200 {'GVNS': {'objective': (23.36240905047277, 1.935156014060852), 'runtime': (0.0657153159995687, 0.003766182238655238)}, 'DL-GVNS': {'objective': (22.320967505686557, 0.015003253848977676), 'runtime': (0.0642537849994369, 0.0026306111

In [6]:
x = np.arange(len(datasets))
width = 0.36
fig, ax = plt.subplots(figsize=(8.0, 4.8))
for offset, method, color in ((-width / 2, "gvns", "#4C78A8"), (width / 2, "dl_gvns", "#54A24B")):
    means = [objective_summary[(dataset, method)][0] for dataset in datasets]
    errors = [objective_summary[(dataset, method)][1] for dataset in datasets]
    ax.bar(x + offset, means, width, yerr=errors, capsize=4, label=labels[method], color=color)
ax.set_title("Paired Heuristic Objective Comparison across Datasets")
ax.set_xlabel("Dataset")
ax.set_ylabel("Conditional beta-mean objective (mean ± SD)")
ax.set_xticks(x, datasets)
ax.legend()
ax.text(0.01, 0.02, "Two paired seeds; 100 scenarios; p=3; alpha=0.5; beta=0.5; budget = 8 evaluations", transform=ax.transAxes, fontsize=8)
fig.tight_layout()
objective_path = OUTPUT_DIR / "gvns_dl_gvns_objective_comparison.png"
fig.savefig(objective_path, dpi=300, bbox_inches="tight")
plt.show()
print(objective_path)

/home/miladkahnooji/Milad/Personal_projects/karlancer/projects/Python math model/Single-allocation-hub-location/outputs/article_presentation/gvns_dl_gvns_objective_comparison.png


/tmp/ipykernel_1170763/228269765.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
fig, ax = plt.subplots(figsize=(8.0, 4.8))
for offset, method, color in ((-width / 2, "gvns", "#4C78A8"), (width / 2, "dl_gvns", "#54A24B")):
    means = [runtime_summary[(dataset, method)][0] for dataset in datasets]
    errors = [runtime_summary[(dataset, method)][1] for dataset in datasets]
    ax.bar(x + offset, means, width, yerr=errors, capsize=4, label=labels[method], color=color)
ax.set_title("Paired Heuristic Runtime Comparison across Datasets")
ax.set_xlabel("Dataset")
ax.set_ylabel("Runtime in seconds (mean ± SD)")
ax.set_xticks(x, datasets)
ax.legend()
ax.text(0.01, 0.02, "Two paired seeds; 100 scenarios; equal evaluation budget = 8", transform=ax.transAxes, fontsize=8)
fig.tight_layout()
runtime_path = OUTPUT_DIR / "gvns_dl_gvns_runtime_comparison.png"
fig.savefig(runtime_path, dpi=300, bbox_inches="tight")
plt.show()
print(runtime_path)

/home/miladkahnooji/Milad/Personal_projects/karlancer/projects/Python math model/Single-allocation-hub-location/outputs/article_presentation/gvns_dl_gvns_runtime_comparison.png


/tmp/ipykernel_1170763/462156703.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Five-method bounded CAB25 comparison and availability

Benders, GVNS, RCBS, DL-GVNS and DL-RCBS are implemented. Benders is a bound-producing exact method when it converges; the displayed CAB25 run is deliberately iteration-bounded. The remaining four methods are heuristics and never proven optimal.

In [8]:
flow_name, distance_name = DATASETS["CAB25"]
matrices = load_matrix_pair(DATA_ROOT / flow_name, DATA_ROOT / distance_name)
flows = generate_flow_scenarios(matrices.flow, scenario_count=100, seed=0)
rank_scores, ranker_identity = load_rank_scores(matrices.distance, matrices.flow, RANKER_CHECKPOINT)
search = GVNSConfig(p=3, alpha=.5, beta=.5, max_iterations=2, max_evaluations=4)
base = solve_gvns(matrices.distance, flows.flows, flows.probabilities, search, 0)
guided = solve_dl_gvns(matrices.distance, flows.flows, flows.probabilities, search, 0, rank_scores, ranker_identity)
cbs = solve_cbs(matrices.distance, matrices.flow, flows.flows, flows.probabilities, 3, .5, .5, max_evaluations=4)
dlcbs = solve_dl_cbs(matrices.distance, matrices.flow, flows.flows, flows.probabilities, 3, .5, .5, scores=rank_scores, max_evaluations=4)
benders = solve_benders(matrices.distance, flows.flows, flows.probabilities, 3, .5, .5, max_iterations=2)
method_records = [("Benders", benders.objective, benders.status), ("GVNS", base.objective, base.status), ("CBS (RCBS)", cbs["objective"], cbs["status"]), ("DL-GVNS", guided.objective, guided.status), ("DL-CBS (DL-RCBS)", dlcbs["objective"], dlcbs["status"])]
fig, ax = plt.subplots(figsize=(11.2, 5.8)); colors = ["#F58518", "#4C78A8", "#54A24B", "#72B7B2", "#E45756"]
bars = ax.bar([x[0] for x in method_records], [x[1] for x in method_records], color=colors)
maximum = max(float(x[1]) for x in method_records)
ax.set_ylim(0, maximum * 1.22)
for bar, (_, _, status) in zip(bars, method_records):
    ax.annotate(status.replace("_", " "),
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 5), textcoords="offset points", ha="center",
                va="bottom", rotation=12, fontsize=7.5)
ax.set_title("CAB25 Five-Method Comparison (100 scenarios; bounded runs)", pad=22)
ax.set_ylabel("Conditional beta-mean objective")
fig.text(.5, .02,
         "Benders is a bounded incumbent with certified gap; no method is claimed optimal.",
         ha="center", fontsize=8.5)
fig.subplots_adjust(top=.84, bottom=.15)
five_method_path = OUTPUT_DIR / "cab25_five_method_comparison.png"
fig.savefig(five_method_path, dpi=300, bbox_inches="tight")
plt.show()
availability = [["Exact binary MILP", "Tiny validation", "Implemented", "Proven only when CBC proves optimality"], ["Benders", "Bounded CAB25 / tiny certification", "Implemented", "Reports LB, UB and absolute/relative gap"], ["GVNS", "CAB25/AP100/AP150/AP200", "Implemented", "Seeded heuristic; never proven optimal"], ["CBS (RCBS)", "CAB25/AP100/AP150/AP200", "Implemented", "Clustered potential-hub heuristic; never proven optimal"], ["DL-GVNS / DL-CBS", "CAB25/AP100/AP150/AP200", "Implemented", "DLHr-guided heuristics; never proven optimal"]]
fig, ax = plt.subplots(figsize=(11.0, 4.1)); ax.axis("off"); table = ax.table(cellText=availability, colLabels=["Method", "Availability", "Status", "Interpretation"], loc="center", cellLoc="left", colWidths=[.2,.27,.17,.36]); table.auto_set_font_size(False); table.set_fontsize(8.5); table.scale(1,1.55)
for column in range(4): table[(0,column)].set_facecolor("#D9EAF7"); table[(0,column)].set_text_props(weight="bold")
ax.set_title("Method Availability and Result Status", pad=14); availability_path = OUTPUT_DIR / "method_availability.png"; fig.savefig(availability_path, dpi=300, bbox_inches="tight"); plt.show(); print(five_method_path, availability_path)


/home/miladkahnooji/Milad/Personal_projects/karlancer/projects/Python math model/Single-allocation-hub-location/outputs/article_presentation/cab25_five_method_comparison.png /home/miladkahnooji/Milad/Personal_projects/karlancer/projects/Python math model/Single-allocation-hub-location/outputs/article_presentation/method_availability.png


/tmp/ipykernel_1170763/4162883368.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1170763/4162883368.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title("Method Availability and Result Status", pad=14); availability_path = OUTPUT_DIR / "method_availability.png"; fig.savefig(availability_path, dpi=300, bbox_inches="tight"); plt.show(); print(five_method_path, availability_path)


## Validation and figure inventory

Every image is opened with Matplotlib after creation. The sensitivity and repeated-pair plots contain heuristic evidence only; the five-method chart annotates each run status. Benders bounds are reported as `absolute_gap = UB - LB` and `relative_gap = (UB - LB) / max(|UB|, 1e-12)`.

In [9]:
paired_records_path = OUTPUT_DIR / "paired_records.json"
paired_records_path.write_text(json.dumps([record.to_dict() for record in records], indent=2), encoding="utf-8")

figure_paths = [beta_path, alpha_path, objective_path, runtime_path, five_method_path, availability_path]
for path in figure_paths:
    pixels = plt.imread(path)
    assert path.is_file() and path.stat().st_size > 0 and pixels.size > 0
    print(f"validated: {path.name} ({path.stat().st_size:,} bytes)")
print(f"validated: {paired_records_path.name} ({paired_records_path.stat().st_size:,} bytes)")
print("Heuristic charts make no optimality claim; Benders is displayed with its bounded status and certified gap.")

validated: cab25_objective_vs_beta.png (168,954 bytes)
validated: cab25_objective_vs_alpha.png (192,437 bytes)
validated: gvns_dl_gvns_objective_comparison.png (118,339 bytes)
validated: gvns_dl_gvns_runtime_comparison.png (107,418 bytes)
validated: cab25_five_method_comparison.png (165,068 bytes)


validated: method_availability.png (153,366 bytes)
validated: paired_records.json (28,037 bytes)
Heuristic charts make no optimality claim; Benders is displayed with its bounded status and certified gap.
